# IT9002 Lab 1.3: Task-Specific Text Preprocessing

**Unit 01: Language, Text, and the NLP Pipeline**

**IT9002 Natural Language Processing**

## Objectives

By the end of this lab, the following should be clear:

- How to select an appropriate preprocessing pipeline for a given task and corpus, with a clear reason for each operation.
- How to measure the effect of preprocessing operations using token and type counts.
- How to use examples of meaning changes to evaluate whether a preprocessing operation should be applied.
- Why some preprocessing operations designed for English do not transfer directly to Arabic.

## Data

- **Sentiment Labelled Sentences** (Kotzias et al., 2015), UCI Machine Learning Repository dataset 331, https://archive.ics.uci.edu/dataset/331/sentiment+labelled+sentences, license CC BY 4.0.
    
    - The file `reviews.csv` (220 KB).

- **GOV.UK guidance pages** on employment, tax and civic rules, collected through the GOV.UK content API, https://www.gov.uk/api/content, Open Government Licence v3.0. 
    
    - The file `policy_docs.csv` (144 KB).

- **Arabic Wikipedia** article introductions, https://ar.wikipedia.org, license CC BY-SA 4.0.
    
    - The file `arabic_sentences.csv` (6 KB).

All files can be downloaded from Moodle. Download them onto your computer, then run the cell below. If a file is already available in the runtime, the cell loads it directly; otherwise, it opens a file-upload dialog.

In [1]:
from pathlib import Path
import pandas as pd

def load(name):
    path = next((p for p in (Path(name), Path("data") / name) if p.exists()), None)
    if path is None:
        from google.colab import files
        files.upload()
        path = Path(name)
    return pd.read_csv(path)

reviews = load("reviews.csv")
policy = load("policy_docs.csv")
arabic = load("arabic_sentences.csv")
print(reviews.shape, policy.shape, arabic.shape)

(3000, 3) (14, 3) (20, 3)


The three files contain **3,000 one-sentence reviews**, **14 documents**, and **20 sentences**. The policy documents and the Arabic sentences were collected from public web pages, and each row records the page it was taken from.

The cell below prints these sources.

In [2]:
print("Policy documents (GOV.UK, Open Government Licence v3.0):")
for title, url in zip(policy["title"], policy["url"]):
    print(f"  {title}: {url}")
print("\nArabic sentences (Arabic Wikipedia, CC BY-SA 4.0):")
for title, url in arabic[["source_title", "source_url"]].drop_duplicates().itertuples(index=False):
    print(f"  {title}: {url}")

Policy documents (GOV.UK, Open Government Licence v3.0):
  Statutory Sick Pay (SSP): https://www.gov.uk/statutory-sick-pay
  Maternity pay and leave: https://www.gov.uk/maternity-pay-leave
  Holiday entitlement: https://www.gov.uk/holiday-entitlement-rights
  Redundancy: your rights: https://www.gov.uk/redundancy-your-rights
  National Minimum Wage and National Living Wage rates: https://www.gov.uk/national-minimum-wage-rates
  Student visa: https://www.gov.uk/student-visa
  Jury service: https://www.gov.uk/jury-service
  How Council Tax works: https://www.gov.uk/council-tax
  Self Assessment tax returns: https://www.gov.uk/self-assessment-tax-returns
  Flexible working: https://www.gov.uk/flexible-working
  Pregnant employees' rights: https://www.gov.uk/working-when-pregnant-your-rights
  Unpaid parental leave: https://www.gov.uk/parental-leave
  Income Tax rates and Personal Allowances: https://www.gov.uk/income-tax-rates
  Employment contracts: https://www.gov.uk/employment-contract

These 14 pages and 8 articles are the complete source of the two collected files.

## Setup

The cell below:

- Downloads the **4** NLTK resources (~17 MB) used in this lab.

- Imports the NLTK tools used later.

- Loads the `en_core_web_sm` pipeline from spaCy, and downloads it first if it is not available.

- Prints the versions of NLTK, spaCy, and pandas.

In [3]:
import re, time
from collections import Counter, defaultdict

import nltk
import spacy

for resource in ("punkt_tab", "stopwords", "wordnet", "averaged_perceptron_tagger_eng"):
    nltk.download(resource, quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

try:
    nlp_spacy = spacy.load("en_core_web_sm")
except OSError:
    spacy.cli.download("en_core_web_sm")
    nlp_spacy = spacy.load("en_core_web_sm")

print("NLTK", nltk.__version__, "| spaCy", spacy.__version__, "| pandas", pd.__version__)

NLTK 3.9.4 | spaCy 3.8.16 | pandas 3.0.2


## 1. Configuring the Preprocessing Pipeline

In this lab, the preprocessing operations are organized into a single function with a **parameter** for each operation. This makes it possible to change one operation at a time and observe its effect.

The cell below defines three functions:

- `preprocess()` returns the tokens of one document after the selected operations are applied.
- `stage_table()` counts the tokens and the types after each operation across a whole corpus.
- `show()` prints a document before and after preprocessing, so that a change in meaning can be observed.

Every investigation in this lab changes the parameters of these functions.

Each parameter of `preprocess()` keeps its default value unless a different value is passed when the function is called. The parameters are applied in the following order:

- `expand` (default `False`): when set to `True`, every word ending in `n't` is rewritten before the text is tokenized, so `didn't` becomes `did not`. The negation then remains as the separate word `not`, which the alphabetic filter would otherwise remove.

- `lowercase` (default `True`): converts every token to lowercase.

- `alpha_only` (default `True`): keeps only the tokens made entirely of letters. Punctuation, numbers, and mixed forms such as `4the` are removed.

- `stop_words` (default: the NLTK English stop-word list): removes every token that appears on the list. Setting it to `None` keeps all tokens.

- `keep` (default: no words): a set of words that are kept even when they appear on the stop-word list, for example `keep={"not", "no"}`.

- `reduce` (default `None`): `"stem"` applies the Porter stemmer, `"lemma"` applies spaCy lemmatization, and `None` leaves the tokens unchanged.

`stage_table()` accepts the same parameters and records the number of tokens and types after each operation. 

`show()` passes any parameter given to it on to `preprocess()`, for example `show(reviews["sentence"], [41], reduce="stem")`.

In [4]:
STOP = set(stopwords.words("english"))
porter, wordnet = PorterStemmer(), WordNetLemmatizer()

def preprocess(text, lowercase=True, alpha_only=True, stop_words=STOP, keep=(), reduce=None, expand=False):
    if expand:
        text = re.sub(r"n't\b", " not", text)
    tokens = word_tokenize(text)
    if lowercase:
        tokens = [t.lower() for t in tokens]
    if alpha_only:
        tokens = [t for t in tokens if t.isalpha()]
    if stop_words:
        tokens = [t for t in tokens if t not in stop_words or t in keep]
    if reduce == "stem":
        tokens = [porter.stem(t) for t in tokens]
    elif reduce == "lemma":
        tokens = [t.lemma_.lower() for t in nlp_spacy(" ".join(tokens))]
    return tokens

def stage_table(texts, lowercase=True, alpha_only=True, stop_words=STOP, keep=(), reduce=None, expand=False):
    if expand:
        texts = [re.sub(r"n't\b", " not", text) for text in texts]
    rows, tokens = [], [t for text in texts for t in word_tokenize(text)]
    rows.append(("Tokens" + (" (contractions expanded)" if expand else ""), tokens))
    if lowercase:
        tokens = [t.lower() for t in tokens]
        rows.append(("Lowercased", tokens))
    if alpha_only:
        tokens = [t for t in tokens if t.isalpha()]
        rows.append(("Alphabetic only", tokens))
    if stop_words:
        tokens = [t for t in tokens if t not in stop_words or t in keep]
        rows.append(("Stop words removed" + (" (negation kept)" if keep else ""), tokens))
    if reduce == "stem":
        tokens = [porter.stem(t) for t in tokens]
        rows.append(("Porter stems", tokens))
    elif reduce == "lemma":
        partial = dict(lowercase=lowercase, alpha_only=alpha_only, stop_words=stop_words, keep=keep)
        tokens = [t.lemma_.lower() for doc in nlp_spacy.pipe((" ".join(preprocess(x, **partial)) for x in texts), batch_size=500) for t in doc]
        rows.append(("spaCy lemmas", tokens))
    return pd.DataFrame([(n, len(t), len(set(t))) for n, t in rows], columns=["stage", "tokens", "types"]).set_index("stage")

def show(texts, ids, **params):
    for i in ids:
        print(f"[{i}] {texts[i]}")
        print(f"     -> {' '.join(preprocess(texts[i], **params))}")

The cell below applies the default parameter values to the first three reviews.

In [5]:
show(reviews["sentence"], [0, 1, 2])

[0] So there is no way for me to plug it in here in the US unless I go by a converter.
     -> way plug us unless go converter
[1] Good case, Excellent value.
     -> good case excellent value
[2] Great for the jawbone.
     -> great jawbone


## 2. Task A: Sentiment Analysis of Reviews

**Task:** The system receives a one-sentence review and classifies it as either **positive** or **negative**. The preprocessing pipeline should preserve the information needed to determine the **sentiment** expressed in the sentence.

**Corpus:** The dataset contains 3,000 one-sentence reviews from three sources: **Amazon product reviews**, **IMDb film reviews**, and **Yelp restaurant reviews**. 

Each review is labeled `1` for positive or `0` for negative.

The cell below prints the number of reviews for each source and label.

In [6]:
print(reviews.groupby(["source", "label"]).size().unstack())

label     0    1
source          
amazon  500  500
imdb    500  500
yelp    500  500


The classes are **balanced** within each source, so neither label is more common by default.

### 2.1 Information Removed by the Default Pipeline

The default pipeline applies case folding, an alphabetic filter, stop-word removal, and Porter stemming.

The cell below summarizes how these preprocessing steps affect the number of tokens and types in the review corpus.

In [7]:
stage_table(reviews["sentence"], reduce="stem")

,tokens,types
stage,,
Tokens,41297,6077
Lowercased,41297,5353
Alphabetic only,34918,4970
Stop words removed,17711,4847
Porter stems,17711,3788


Stop-word removal reduces the corpus from **34,918 tokens** to **17,711 tokens**, approximately 1/2 of the original. Stemming reduces the vocabulary from **4,847 types** to **3,788 types**.

For sentiment analysis, **negation** is important because it can reverse the meaning of a statement. Removing negation words may remove information needed to distinguish positive from negative sentiment.

The cell below counts how many reviews contain a negation word:

- **Before** preprocessing.
- **After** applying the default pipeline.

In [8]:
NEGATION = {"not", "n't", "no", "never", "nor", "nothing", "neither", "cannot", "without", "hardly"}

def has_negation(tokens):
    return any(t in NEGATION for t in tokens)

before = reviews["sentence"].apply(lambda s: has_negation(word_tokenize(s.lower())))
after = reviews["sentence"].apply(lambda s: has_negation(preprocess(s)))
print(f"sentences with a negation word: {before.sum()} ({before.mean():.0%} of the corpus)")
print(f"of which negative: {reviews[before]['label'].eq(0).sum()}, positive: {reviews[before]['label'].eq(1).sum()}")
print(f"still containing one after the default pipeline: {after.sum()}")

sentences with a negation word: 691 (23% of the corpus)
of which negative: 532, positive: 159
still containing one after the default pipeline: 93


**691 reviews** (23% of the corpus) contain a negation word, and **532** of these reviews are negative. After applying the default pipeline, only **93 reviews** still contain a negation word, because the stop-word list and the alphabetic filter remove the most common negation forms.

For example, the review `I was not impressed` is reduced to `impress`. The preprocessing pipeline has removed the negation and changed the meaning of the original review.

The cell below shows three negative reviews before and after applying the default pipeline.

In [9]:
show(reviews["sentence"], [41, 142, 455], reduce="stem")

[41] I was not impressed by this product.
     -> impress product
[142] I was not happy with this item.
     -> happi item
[455] Not good when wearing a hat or sunglasses.
     -> good wear hat sunglass


### 2.2 Selecting a Preprocessing Pipeline for Sentiment

Choose the preprocessing pipeline for this task by adjusting the parameters of `preprocess()`.

The selected pipeline should satisfy two requirements:

- All **691 reviews** that originally contain a negation word should still contain a negation after preprocessing, so `after.sum()` should return **at least 691**.
- After the reduction step, the vocabulary should contain **no more than 4,200 types**.

Then provide evidence for the chosen pipeline:

- The stage table showing the effect of each selected operation.
- **3 reviews** where the default pipeline changed the meaning but the chosen pipeline preserved it.
- The completed decision table.

In [10]:
# TODO: Choose the parameters (keep, reduce, and any others), then print the stage table and the negation count.


In [11]:
# TODO: Print 3 reviews where the default pipeline reversed the meaning and the chosen pipeline did not, in both versions.


Complete the table below with **one row** for each preprocessing operation.

In the **Reason** column, support each decision with evidence from either:

- A count reported in the stage table.
- One of the review examples examined above.

| Operation | Choice | Reason |
|---|---|---|
| Case folding | | |
| Alphabetic filter | | |
| Stop words | | |
| Stemming or lemmatization | | |

## 3. Task B: Keyword Search in Policy Documents

**Task:** The system provides a search function over a collection of guidance documents. A document matches a query when all query terms are found in that document.

The preprocessing pipeline determines what counts as a match. For example, it affects whether `paying` matches `pay`, `dependants` matches `dependant`, and `Tax` matches `tax`.

**Corpus:** The dataset contains 14 GOV.UK guidance pages with **25,691 words** in total. Individual documents contain 425-5,446 words each.

The cell below lists the documents and their lengths.

In [12]:
policy["words"] = policy["text"].str.split().str.len()
policy[["title", "words"]]

,title,words
0,Statutory Sick Pay (SSP),715
1,Maternity pay and leave,1059
2,Holiday entitlement,2424
3,Redundancy: your rights,2123
4,National Minimum Wage and National Living Wage...,425
5,Student visa,5446
6,Jury service,3594
7,How Council Tax works,2044
8,Self Assessment tax returns,4035
9,Flexible working,1121


### 3.1 Exact Term Matching

An **index** maps each token to the documents in which it appears.

The cell below:

- Builds the index without stop-word removal or stemming, so each query term matches only the same form in the documents.
- Defines `search()`, which returns documents containing every term in a query.
- Runs 9 queries and reports how many documents match each one.

In [13]:
def build_index(texts, **params):
    index = defaultdict(set)
    for i, text in enumerate(texts):
        for token in preprocess(text, **params):
            index[token].add(i)
    return index

def search(query, index, **params):
    terms = preprocess(query, **params)
    sets = [index.get(t, set()) for t in terms]
    return sorted(set.intersection(*sets)) if sets else []

QUERIES = ["sick pay", "holiday entitlement", "paying taxes", "claiming", "dependants",
           "employers", "apprentices wage", "contract terms", "dismissed"]

def query_table(index, **params):
    return pd.DataFrame({"documents found": {q: len(search(q, index, **params)) for q in QUERIES}})

exact_params = dict(stop_words=None, reduce=None)
index_exact = build_index(policy["text"], **exact_params)
print(f"vocabulary: {len(index_exact):,} types")
query_table(index_exact, **exact_params)

vocabulary: 1,881 types


,documents found
sick pay,3
holiday entitlement,2
paying taxes,0
claiming,1
dependants,2
employers,6
apprentices wage,1
contract terms,4
dismissed,4


The index contains **1,881 types**. With exact matching, the query `paying taxes` returns no documents, even though 7 of the 14 documents discuss paying tax. Similarly, the query `claiming` returns only 1 document, although 7 documents discuss claims.

The problem is that the query and the documents may use different grammatical forms of the same word. Exact matching treats these forms as different terms, which can cause relevant documents to be **missed**.

The cell below lists the documents returned by two example queries.

In [14]:
titles = policy["title"]
print("sick pay:", [titles[i] for i in search("sick pay", index_exact, **exact_params)])
print("dependants:", [titles[i] for i in search("dependants", index_exact, **exact_params)])

sick pay: ['Statutory Sick Pay (SSP)', 'Holiday entitlement', 'Employment contracts']
dependants: ['Redundancy: your rights', 'Student visa']


### 3.2 Selecting a Preprocessing Pipeline for Search

Choose the preprocessing parameters for the search index by selecting:

- A reduction method: `reduce="stem"` or `reduce="lemma"`.
- Whether stop words should be removed.

Rebuild the index for each parameter value and rerun the queries.

The selected pipeline should satisfy two requirements:

- `paying taxes` should return **7 documents**, and `dependants` should return **8 documents**.
- The vocabulary of the index should contain **fewer than 1,400 types**.

Then provide evidence for the selected pipeline:

- The query results for each parameter value.
- **3 stems** from the index that combine words with different meanings.
- The completed decision table.

In [15]:
# TODO: Build the index with reduce="stem" and with reduce="lemma", then print the vocabulary size and the query table for each.


In [16]:
# TODO: Map each stem to the set of document words that produce it, print the 10 largest groups, then name 3 wrong merges.


Complete the decision table below.

| Operation | Choice | Reason |
|---|---|---|
| Case folding | | |
| Alphabetic filter | | |
| Stop words | | |
| Stemming or lemmatization | | |

## Exercises

### Exercise 1: Evaluating the English Preprocessing Pipeline on Arabic Text

Apply `preprocess()` with its default parameter values to the 20 Arabic sentences and examine how each operation behaves.

Report:

- How many tokens are changed by case folding.
- How many tokens are removed by the alphabetic filter, and which tokens are removed.
- How many tokens are removed by the English stop-word list.
- How many tokens are changed by the Porter stemmer.

Then process the same sentences with the English spaCy pipeline and count the part-of-speech categories it assigns.

Finish with a Markdown cell explaining which preprocessing operations are unsuitable for Arabic and why.

In [17]:
# TODO: Report the per-operation counts on the Arabic sentences and the spaCy tag counts, then explain the results in a Markdown cell.


**Expected results:**

- **Case folding** changes 0 tokens because Arabic does not use letter case.
- **The English stop-word list** removes 0 tokens, and **the Porter stemmer** changes 0 tokens because both are designed for English text.
- **The alphabetic filter** removes approximately 80 of the 359 tokens, including many valid Arabic words. This happens because the tokenizer may leave the Arabic comma `،` attached to the preceding word. Words containing diacritics may also fail `isalpha()` because the diacritic is stored as a separate Unicode character attached to the letter.
- **The English tagger** assigns approximately 70% of the tokens to the proper-noun category, reflecting its poor fit for Arabic text that it was not trained to analyze.

The second part replaces three unsuitable English-specific operations with Arabic-compatible alternatives available in NLTK:

- A regular-expression pattern that removes Arabic diacritics using the Unicode ranges U+064B to U+0652 and U+0670.
- The Arabic stop-word list.
- The ISRI Arabic stemmer.

Report:

- The number of types before and after removing diacritics.
- The number of tokens removed by the Arabic stop-word list.
- **5 Arabic words** alongside their ISRI stems, including one example where the stemmer produces an unsuitable result.

In [18]:
# TODO: Remove the diacritics, apply the Arabic stop-word list, apply the ISRI stemmer, and print the counts.


**Expected results:**

- Removing diacritics and attached commas reduces the vocabulary by approximately 10 types, from 267 to about 255.
- The Arabic stop-word list removes approximately 60 of the 359 tokens.
- The ISRI stemmer reduces many words toward a three-letter root. For `المحادثة` (conversation), it correctly produces `حدث`. However, it can also produce misleading results for proper names: `البحرين` (Bahrain) becomes `بحر` (sea), and `المنامة` (Manama) becomes `نام`.

Using a stemmer designed for Arabic improves the linguistic fit of the preprocessing pipeline, but its output still needs to be evaluated according to the task and the information that should be preserved.

### Exercise 2: Examining Letter Case in Named Entities

Named entity extraction often relies on the original letter case because capitalization can help distinguish names from ordinary words.

Using the policy documents:

- Count how often each word appears with an initial capital letter and how often it appears in lowercase.
- Print the **10 words** with the highest capitalized counts that also appear in lowercase at least 5 times.
- In a Markdown cell, identify which of these words refer to names, such as an office, scheme, or document, when capitalized but have an ordinary meaning when written in lowercase.

In [19]:
# TODO: Count the capitalized and lowercase forms of every word, print the 10 most frequent capitalized words that also occur in lowercase, then write the Markdown reading.


**Expected result:** `Tax`, `Self`, `Council`, `Allowance`, `Leave`, and `Pay` appear among the most frequent capitalized forms.

Several of these words have different roles depending on their capitalization:

- `Self Assessment` is the name of a tax system, while `self` is used as an ordinary pronoun.
- `Council Tax` is the name of a specific charge, while `council` refers more generally to a council.
- `Leave` in `Statutory Maternity Leave` is part of the name of an entitlement, while `leave` can also be used as a verb.
- `Pay` in `Statutory Sick Pay` is part of a named entitlement, while `pay` can also refer to the act of paying.

Case folding removes these capitalization differences. As a result, a named entity system applied after case folding loses information that may help distinguish named entities from ordinary uses of the same words.

### Exercise 3 (Extension): Preserving Hyphenated Terms in Search

The default preprocessing pipeline can remove information that is useful for search. For example, the alphabetic filter removes terms such as `self-employed` and `full-time` because they contain hyphens, while case folding converts abbreviations such as `SSP` to lowercase.

For search, these forms may need to be preserved so that queries can match the corresponding terms in the documents.

Complete the following steps:

- Write a regular-expression tokenizer that keeps hyphenated words as single tokens.
- Build a new search index using this tokenizer.
- Compare the results of the queries `self-employed`, `full-time`, and `ssp` using the default index and the new index.

In [20]:
# TODO: Write a regular-expression tokenizer that keeps hyphenated words, build the index with it, and compare the 3 queries under both indexes.


**Expected result:** The default index returns no documents for `self-employed` or `full-time` because the alphabetic filter removes tokens that contain hyphens. The regular-expression index preserves these terms and finds approximately 4 matching documents for each query.

Both indexes return results for `ssp` because case folding changes `SSP` to `ssp` without removing the letters.

This demonstrates that tokenization is an important preprocessing decision. When a corpus contains meaningful hyphenated terms, the tokenizer should be designed to preserve them.